In [1]:
using Plots
using Revise
using DataInterpolations
using RegularizationTools
using NaturalNeighbours

In [2]:
function read_custom_file(filepath::String)
    array1 = Float64[]
    array2 = Float64[]

    start_reading = false

    num_regex = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    open(filepath, "r") do file
        for line in eachline(file)

            clean_line = strip(line)

            if startswith(clean_line, "+") || startswith(clean_line, "-")
                matches = [m.match for m in eachmatch(num_regex, clean_line)]

                if length(matches) >= 2
                    push!(array1, parse(Float64, matches[1]))
                    push!(array2, parse(Float64, matches[2]))
                end
            else
                continue
            end
        end
    end
    return array1, array2
end

read_custom_file (generic function with 1 method)

In [3]:
cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/2.S9272.PtCoCu.7/S9272-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/6.S9281.CoPt.10/S9281-FORC-50-2000-5s"
#cale = "/home/lali/TITAN-ROG-sync/julia/_Magnetism/4.S9283.CoPt.6/S9283-FORC-50-1500-5s"
H_read = Float64[]
M_read = Float64[]
H_read, M_read = read_custom_file(cale * ".txt")

([1926.149, 1966.079, 2006.295, 1845.812, 1900.373, 1954.284, 2006.217, 1765.163, 1819.753, 1873.858  …  1516.857, 1570.778, 1625.594, 1679.432, 1733.308, 1788.094, 1841.95, 1895.938, 1950.765, 2004.618], [0.0002315941, 0.0002317994, 0.0002326804, 0.0002304468, 0.0002313527, 0.0002317877, 0.0002337235, 0.0002295009, 0.0002304182, 0.0002303912  …  0.0002133032, 0.0002193848, 0.0002233787, 0.0002226158, 0.0002270869, 0.000226353, 0.0002305136, 0.0002307518, 0.0002289827, 0.0002341819])

In [4]:
#Normalize
normH = 1.0e3
normM = 1.0e-3
H_read = H_read ./ normH
M_read = M_read ./ normM
H_read, M_read

([1.926149, 1.966079, 2.006295, 1.845812, 1.900373, 1.9542840000000001, 2.006217, 1.765163, 1.819753, 1.873858  …  1.516857, 1.570778, 1.625594, 1.679432, 1.733308, 1.788094, 1.84195, 1.8959380000000001, 1.950765, 2.004618], [0.2315941, 0.23179940000000002, 0.2326804, 0.2304468, 0.2313527, 0.23178769999999999, 0.2337235, 0.2295009, 0.2304182, 0.2303912  …  0.21330319999999997, 0.2193848, 0.2233787, 0.2226158, 0.22708689999999998, 0.226353, 0.23051359999999999, 0.23075179999999998, 0.22898269999999998, 0.2341819])

In [5]:
scatter(H_read, M_read, 
    title = "Scatter Plot of Data", 
    xlabel = "H", 
    ylabel = "M", 
    legend = false,   
    markersize = 3, 
    markercolor = :indianred,
    markerstrokecolor =:indianred
)

In [6]:
# Detect FORCs (Hr, numPointsPerFORC)
Hr_read = Float64[]
numPointsPerFORC = Int[]

current_Hr = H_read[1]
push!(Hr_read, current_Hr)

counterPointsPerFORC = 1

for i in 2:(length(H_read))
    if H_read[i] < H_read[i-1]
        global current_Hr = H_read[i]
        push!(numPointsPerFORC, counterPointsPerFORC)
        global counterPointsPerFORC = 0
    end
    global counterPointsPerFORC += 1
    push!(Hr_read, current_Hr)
end
push!(numPointsPerFORC, counterPointsPerFORC)
numFORCs = length(numPointsPerFORC)

println("----------- Total $(length(numPointsPerFORC)) FORCs -----------")
println("---------  $(numPointsPerFORC[1]) first / $(numPointsPerFORC[numFORCs]) last ---------")

----------- Total 50 FORCs -----------
---------  3 first / 75 last ---------


In [7]:
# Save Hr-H-M original file
n = length(Hr_read)
if length(H_read) != n || length(M_read) != n
    error("Error: All three arrays must have the same length.")
end

open(cale * "_Hr-H-M_orig.dat", "w") do file
    for i in 1:n
        println(file, "$(Hr_read[i])  $(H_read[i])  $(M_read[i])")
    end
end

In [8]:
function gimmeOneFORC(theFORC::Int64)
    contorNumPoints = 0
    H_interp = Float64[]
    M_interp = Float64[]
    #detect indexes for $(theFORC)
    if theFORC > 1
        for i in 1:theFORC-1
            contorNumPoints += numPointsPerFORC[i]
        end
    else
        contorNumPoints = 0
    end

    #myHr = Hr_read[contorNumPoints]
    startPointOnFORC = contorNumPoints + 1
    push!(H_interp, H_read[startPointOnFORC])
    push!(M_interp, M_read[startPointOnFORC])
    contorNumPoints = 1
    while (Hr_read[startPointOnFORC+contorNumPoints-1] - Hr_read[startPointOnFORC+contorNumPoints]) < 1.0e-5 #eps - compare floats
        push!(H_interp, H_read[startPointOnFORC+contorNumPoints])
        push!(M_interp, M_read[startPointOnFORC+contorNumPoints])
        contorNumPoints += 1
        if (startPointOnFORC + contorNumPoints) > length(Hr_read)
            break
        end
    end
    H_interp, M_interp
end

gimmeOneFORC (generic function with 1 method)

In [9]:
showTest = true

true

In [10]:
if (showTest)
    # Interpolate one FORC
    plotInterpFORC = numFORCs
    println("----------------- Interpolating Akima $(plotInterpFORC)-th FORC ----------------")
    H_interp, M_interp = gimmeOneFORC(plotInterpFORC)
    Example = AkimaInterpolation(M_interp, H_interp)
    plot(Example)
end

----------------- Interpolating Akima 50-th FORC ----------------


In [11]:
if (showTest)
    # Smooth one FORC (N - num points after smoothing)
    t, u = gimmeOneFORC(plotInterpFORC)
    d = 3 + div(plotInterpFORC, 3) - 1
    A = RegularizationSmooth(u, t, d; alg=:gcv_svd)
    û = A.û
    N = plotInterpFORC #length(t)
    titp = collect(range(minimum(t), maximum(t), length=N))
    uitp = A.(titp)
    scatter(t, u, label="simulated data", legend=:top)
    plot!(titp, uitp, label="smoothed interpolation")
end

In [12]:
if (showTest)
    # Save interpolated / smoothed file for a single FORC
    nSmooth = length(titp)
    nOrig = length(H_interp)
    n = maximum([nSmooth, nOrig])

    open(cale * "_InterpSmoothCheck.dat", "w") do file
        for i in 1:n
            if (nSmooth >= nOrig)
                if (i <= nOrig)
                    println(file, "$(t[i]), $(u[i]), $(H_interp[i]), $(M_interp[i]), $(titp[i]), $(uitp[i])")
                else
                    println(file, "       ,        ,              ,                , $(titp[i]), $(uitp[i])")
                end
            else
                if (i <= nSmooth)
                    println(file, "$(t[i]), $(u[i]), $(H_interp[i]), $(M_interp[i]), $(titp[i]), $(uitp[i])")
                else
                    println(file, "$(t[i]), $(u[i]), $(H_interp[i]), $(M_interp[i]),           ,           ")
                end
            end
            
        end
    end
end

In [13]:
numFORCs = length(numPointsPerFORC) + 1 # +1 pentru one-point-FORC

M_NoExtend  = zeros(Float64, numFORCs, numFORCs) #matricea M
M_ExtendLine = zeros(Float64, numFORCs, numFORCs) #matricea M
M_ExtendMirror = zeros(Float64, numFORCs, numFORCs) #matricea M
Hr_NoExtend = zeros(Float64, numFORCs)
H_NoExtend  = zeros(Float64, numFORCs)

H, M = gimmeOneFORC(1)
Hr_NoExtend[1] = H[end]
H_NoExtend[numFORCs] = H[end]
M_NoExtend[1, numFORCs] = M[end]
M_ExtendLine[1, numFORCs] = M[end]
M_ExtendMirror[1, numFORCs] = M[end]
for i in 1:numFORCs-1
    M_ExtendLine[1, i] = M[end]
    M_ExtendMirror[1, i] = M[end]
end

for i in 2:numFORCs
    H, M = gimmeOneFORC(i-1)
    Hr_NoExtend[i] = H[begin]
    H_NoExtend[numFORCs-i+1] = H[begin]
end

In [14]:
for i in 2:numFORCs
    t, u = gimmeOneFORC(i-1)
    if t[end] < Hr_NoExtend[1]
        t[end] = Hr_NoExtend[1]
    end

    #Example = AkimaInterpolation(M_interp, H_interp)

    d = 3 + div(i, 3)-1
    A = RegularizationSmooth(u, t, d; alg=:gcv_svd)
    û = A.û
    N = i #length(t)
    titp = [Hr_NoExtend[j] for j=i:-1:1] #collect(range(minimum(t), maximum(t), length=N))
    uitp = A.(titp)
    #println("---------- $(i)/$(numFORCs) -------------")

    for j in 1:i
        M_NoExtend[i, numFORCs-i+j] = uitp[j]
        M_ExtendLine[i, numFORCs-i+j] = uitp[j]
        M_ExtendMirror[i, numFORCs-i+j] = uitp[j]
        #println(" $(M_NoExtend[i, numFORCs-i+j]) ")
    end
    for j in 1:numFORCs-i
        M_ExtendLine[i, j] = uitp[begin]
    end
    for j in numFORCs-i+1:-1:1
        if j+(2*(numFORCs-i+1-j)+1) <= numFORCs
            M_ExtendMirror[i, j] = M_ExtendMirror[i, j+(2*(numFORCs-i+1-j)+1)]
        else
            M_ExtendMirror[i, j] = uitp[end]
        end
    end

end

In [15]:
if (showTest)
    heatmap(M_ExtendMirror, yflip=true)
end

In [16]:
if (showTest)
        scatter(H_NoExtend, transpose(M_ExtendLine), legend=false,
                                                        markersize=3,
                                                        markercolor=:indianred,
                                                        markerstrokecolor=:indianred)
end

In [17]:
open(cale * ".MGRID.REGULAR.dat", "w") do file
    for i in 1:numFORCs
        for j in numFORCs-i+1:numFORCs
            println(file, "$(Hr_NoExtend[i])  $(H_NoExtend[j])  $(M_NoExtend[i, j])")
        end
        print(file, "\n")
    end
end

In [18]:
open(cale * ".MGRID.PrelungLine.dat", "w") do file
    for i in 1:numFORCs
        for j in 1:numFORCs
            println(file, "$(Hr_NoExtend[i])  $(H_NoExtend[j])  $(M_ExtendLine[i, j])")
        end
        print(file, "\n")
    end
end

In [19]:
SP = 4
FORC_NoExtend = zeros(Float64, numFORCs, numFORCs) #matricea FORC 
;

In [20]:
itp = interpolate(Hr_read, H_read, M_read; derivatives=true)

Natural Neighbour Interpolant
    z: [0.2315941, 0.23179940000000002, 0.2326804, 0.2304468, 0.2313527, 0.23178769999999999, 0.2337235, 0.2295009, 0.2304182, 0.2303912  …  0.21330319999999997, 0.2193848, 0.2233787, 0.2226158, 0.22708689999999998, 0.226353, 0.23051359999999999, 0.23075179999999998, 0.22898269999999998, 0.2341819]
    ∇: [(-0.006378042313743983, -0.0009933294054231992), (-0.00205750968866745, 0.012076667757097866), (-0.0027251395017919926, 0.03710819672867521), (-0.006304170922978218, 0.016845853192973232), (0.0019479238573703746, 0.011468843980619524), (0.0021661443672128158, 0.019011779675329844), (0.003252132480383432, 0.087877906994345), (-0.0019302410853399134, 0.01820133906787646), (-0.0022091358294563687, 0.011953116242949135), (-0.00531127560516335, 0.005886250759856655)  …  (0.023635896183142244, 0.08032516371586185), (-0.0056888587086018835, 0.0798020647565623), (-0.026138737435411546, 0.04016673247289281), (0.028235696158935154, 0.02965276076666001), (-0.012925

In [21]:
H_ForInterp = Float64[]
Hr_ForInterp = Float64[]

for i in 1:numFORCs
    for j in numFORCs-i+1:numFORCs
        push!(Hr_ForInterp, Hr_NoExtend[i])
        push!(H_ForInterp, H_NoExtend[j])
    end
end

In [22]:
sibson_1_vals = itp(Hr_ForInterp, H_ForInterp; method=Sibson(1));
;

In [23]:
open(cale * ".INTERP.dat", "w") do file
    for i in 1:length(Hr_ForInterp)
        if (i>2) && (Hr_ForInterp[i] < Hr_ForInterp[i-1])
            println(file, "")
        end
        println(file, "$(Hr_ForInterp[i])  $(H_ForInterp[i])  $(sibson_1_vals[i])")
        

    end
end